### Converting Tunisair_delay.db into CSV


In [ ]:
import sqlite3
import pandas as pd
import os

# Export the first available table from the delay DB to CSV
# Adjust table selection if your DB uses a different table name

# Paths
_db_path = "data/db/tunisair_delay.db"
_db_export_path = "data/csv/tunisair_delay_export.csv"

os.makedirs("data/csv", exist_ok=True)

# Discover tables
conn = sqlite3.connect(_db_path)
_tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';",
    conn,
)['name'].tolist()
print(f"Tables found in DB: {_tables}")

if not _tables:
    conn.close()
    raise RuntimeError("No tables found in the SQLite DB; cannot export.")

# Prefer a table named 'delays' if it exists, otherwise use the first table
_table_name = "delays" if "delays" in _tables else _tables[0]

df_delay = pd.read_sql_query(f"SELECT * FROM {_table_name};", conn)
conn.close()

df_delay.to_csv(_db_export_path, index=False, encoding="utf-8")
print(f"Exported table '{_table_name}' to {_db_export_path} with shape {df_delay.shape}")

### Filter tunisair_delay_export.csv by AIRLINE == 'TU' and export

In [ ]:
# Filter tunisair_delay_export.csv by AIRLINE == 'TU' and export
import pandas as pd

input_file = 'data/csv/tunisair_delay_export.csv'
output_file = 'data/csv/tunisair_delay_export_TU.csv'

try:
    df = pd.read_csv(input_file, encoding='utf-8')
    if 'AIRLINE' not in df.columns:
        raise KeyError("Required column 'AIRLINE' not found in the CSV")

    df_tu = df[df['AIRLINE'] == 'TU']
    df_tu.to_csv(output_file, index=False, encoding='utf-8')

    print(f" Filtered TU records: {len(df_tu)} / {len(df)}")
    print(f" Saved filtered CSV to: {output_file}")
    print("\nPreview:")
    display(df_tu.head(10))
except Exception as e:
    print(f"Error filtering file '{input_file}': {e}")
